In [1]:
from pathlib import Path
import os
import sys
from torchvision.transforms import transforms

In [2]:
ROOT_DIR = Path.cwd().parent
ROOT_DIR

PosixPath('/home/chiendoan/Code_Module6/YOU_ONLY_LOOK_ONCE_YOLO_V1_V2')

In [3]:
sys.path.append(str(ROOT_DIR))

In [4]:
train_dir = ROOT_DIR/'data'/'train'

In [5]:
_dir = Path(ROOT_DIR/'data'/'train'/'VOCdevkit'/'VOC2012'/'ImageSets'/'Main')
# Using set datatype to collect single value
CLASSES = sorted({
    f.rsplit('_',1)[0]
    for f in os.listdir(_dir)
    if '_' in f and f.endswith('.txt')
})
CLASSES

['aeroplane',
 'bicycle',
 'bird',
 'boat',
 'bottle',
 'bus',
 'car',
 'cat',
 'chair',
 'cow',
 'diningtable',
 'dog',
 'horse',
 'motorbike',
 'person',
 'pottedplant',
 'sheep',
 'sofa',
 'train',
 'tvmonitor']

In [6]:
CLASS2INDEX = { clss: (index) for index, clss in enumerate(CLASSES)}
CLASS2INDEX

{'aeroplane': 0,
 'bicycle': 1,
 'bird': 2,
 'boat': 3,
 'bottle': 4,
 'bus': 5,
 'car': 6,
 'cat': 7,
 'chair': 8,
 'cow': 9,
 'diningtable': 10,
 'dog': 11,
 'horse': 12,
 'motorbike': 13,
 'person': 14,
 'pottedplant': 15,
 'sheep': 16,
 'sofa': 17,
 'train': 18,
 'tvmonitor': 19}

In [7]:
from src.data.orgainize_data import *

In [8]:
import timm
model = timm.create_model('resnet18.a3_in1k', pretrained=True)
model.pretrained_cfg

{'url': 'https://github.com/huggingface/pytorch-image-models/releases/download/v0.1-rsb-weights/resnet18_a3_0-40c531c8.pth',
 'hf_hub_id': 'timm/resnet18.a3_in1k',
 'architecture': 'resnet18',
 'tag': 'a3_in1k',
 'custom_load': False,
 'input_size': (3, 160, 160),
 'test_input_size': (3, 224, 224),
 'fixed_input_size': False,
 'interpolation': 'bicubic',
 'crop_pct': 0.95,
 'test_crop_pct': 0.95,
 'crop_mode': 'center',
 'mean': (0.485, 0.456, 0.406),
 'std': (0.229, 0.224, 0.225),
 'num_classes': 1000,
 'pool_size': (5, 5),
 'first_conv': 'conv1',
 'classifier': 'fc',
 'license': 'apache-2.0',
 'origin_url': 'https://github.com/huggingface/pytorch-image-models',
 'paper_ids': 'arXiv:2110.00476'}

In [9]:
trainDataset = VOCdataset(
    data_dir=str(train_dir),
    class2idx=CLASS2INDEX,
    list_transformation=[transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std =(0.229, 0.224, 0.225)
    )],
)

In [10]:
from model.YOLODetector import *

In [11]:
detector = YOLOv2()

In [12]:
state_dict = torch.load(str(ROOT_DIR/'model_weight.pth'))

In [13]:
detector.load_state_dict(state_dict)

<All keys matched successfully>

In [14]:
detector.fit(
    dataset=trainDataset,
    N_epochs=100,
    lr=0.001,
    batch_size=32,
    lamda_noobj=0.1
)

Epoch 1/100 [Train]: 100%|██████████| 179/179 [02:37<00:00,  1.14it/s, avg_loss=1.5783, loss=2.7037, size=768x768]


Epoch 1/100 - train_loss: 1.5783


Epoch 2/100 [Train]: 100%|██████████| 179/179 [02:34<00:00,  1.16it/s, avg_loss=2.0753, loss=2.4744, size=672x672]


Epoch 2/100 - train_loss: 2.0753


Epoch 3/100 [Train]: 100%|██████████| 179/179 [02:51<00:00,  1.05it/s, avg_loss=2.1708, loss=3.5129, size=608x608]


Epoch 3/100 - train_loss: 2.1708


... (190 outputs left out)

Epoch 99/100 [Train]: 100%|██████████| 179/179 [02:30<00:00,  1.19it/s, avg_loss=0.3412, loss=0.3980, size=736x736]


Epoch 99/100 - train_loss: 0.3412


Epoch 100/100 [Train]: 100%|██████████| 179/179 [02:32<00:00,  1.18it/s, avg_loss=0.3240, loss=0.3509, size=672x672]

Epoch 100/100 - train_loss: 0.3240


In [15]:
WEIGHT_PATH = str(ROOT_DIR/'model_weight.pth')
torch.save(detector.state_dict(),WEIGHT_PATH)